In [46]:
from pyspark.sql import SparkSession

# Question 1: Install Spark and PySpark

In [47]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [48]:
# Check Spark version
spark.version

'4.1.1'

# Question 2: Yellow November 2025

In [49]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-04 01:11:48--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2822:b600:b:20a5:b140:21, 2600:9000:2822:f000:b:20a5:b140:21, 2600:9000:2822:d600:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2822:b600:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.1’

yellow_tripdata_202 100%[===================>]  67,84M  58,7MB/s    in 1,2s    

2026-03-04 01:11:50 (58,7 MB/s) - ‘yellow_tripdata_2025-11.parquet.1’ saved [71134255/71134255]



In [50]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [52]:
# Repartition the Dataframe to 4 partitions and save it to parquet.

df.repartition(4).write.parquet('yellow_tripdata_2025-11_repartitioned.parquet')

In [53]:
!ls -lh yellow_tripdata_2025-11_repartitioned.parquet/*

-rw-r--r--@ 1 cristian  staff     0B Mar  4 01:12 yellow_tripdata_2025-11_repartitioned.parquet/_SUCCESS
-rw-r--r--@ 1 cristian  staff    24M Mar  4 01:12 yellow_tripdata_2025-11_repartitioned.parquet/part-00000-2d9db9df-db31-4655-b66e-d4bf5bc30562-c000.snappy.parquet
-rw-r--r--@ 1 cristian  staff    24M Mar  4 01:12 yellow_tripdata_2025-11_repartitioned.parquet/part-00001-2d9db9df-db31-4655-b66e-d4bf5bc30562-c000.snappy.parquet
-rw-r--r--@ 1 cristian  staff    24M Mar  4 01:12 yellow_tripdata_2025-11_repartitioned.parquet/part-00002-2d9db9df-db31-4655-b66e-d4bf5bc30562-c000.snappy.parquet
-rw-r--r--@ 1 cristian  staff    24M Mar  4 01:12 yellow_tripdata_2025-11_repartitioned.parquet/part-00003-2d9db9df-db31-4655-b66e-d4bf5bc30562-c000.snappy.parquet


# Question 3: Count records

In [54]:
# How many taxi trips were there on the 15th of November?
df_filtered = df.filter(df.tpep_pickup_datetime >= '2025-11-15 00:00:00').filter(df.tpep_pickup_datetime < '2025-11-16 00:00:00')
count = df_filtered.count()
print(count)

162604


# Question 4: Longest trip

In [55]:
# Use the function datediff to calculate the difference in hours between the pickup and dropoff datetime columns.
from pyspark.sql.functions import datediff, col

df_with_diff = df.withColumn('diff_hours', (col('tpep_dropoff_datetime') - col('tpep_pickup_datetime')))
df_with_diff.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|          diff_hours|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              

In [56]:
# Longest trip in hours
from pyspark.sql.functions import max

max_diff = df_with_diff.agg(max('diff_hours')).collect()[0][0]
print(max_diff.total_seconds() / 3600)

90.64666666666666


In [57]:
# Another method using SQL
df.registerTempTable('trips')

result = spark.sql("""
    SELECT MAX(TIMESTAMPDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime))/60.0 AS diff_hours
    FROM trips
""")
result.show()

/Users/cristian/Repositories/de-zoomcamp-2026/06-batch/.venv/lib/python3.12/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


+----------+
|diff_hours|
+----------+
| 90.633333|
+----------+



# Question 5: User Interface

In [58]:
# Spark's User Interface which shows the application's dashboard runs on which local port?
spark.sparkContext.uiWebUrl.split(':')[-1]

'4040'

# Question 6: Least frequent pickup location zone

In [59]:
# Download the taxi zone lookup data which contains the mapping of location IDs to zone names and boroughs.
! wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-04 01:12:22--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2822:b600:b:20a5:b140:21, 2600:9000:2822:f000:b:20a5:b140:21, 2600:9000:2822:d600:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2822:b600:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12,04K  --.-KB/s    in 0s      

2026-03-04 01:12:22 (181 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [64]:
# Load the taxi zone lookup data into a temp view on Spark
import pandas as pd
taxi_zone_lookup = pd.read_csv('taxi_zone_lookup.csv')
taxi_zone_lookup_df = spark.createDataFrame(taxi_zone_lookup)
taxi_zone_lookup_df.createOrReplaceTempView('taxi_zone_lookup')

# Join the trips table with the taxi zone lookup table to get the zone names for the pickup and dropoff locations.
df.createOrReplaceTempView('trips')
taxi_zone_lookup_df.createOrReplaceTempView('taxi_zone_lookup')
result = spark.sql("""
    WITH pickup_zones AS (
        SELECT 
            pu_zone.Zone AS pickup_zone
        FROM trips
        JOIN taxi_zone_lookup AS pu_zone ON trips.PULocationID = pu_zone.LocationID
    )
    SELECT 
        pickup_zones.pickup_zone,
        COUNT(*) AS trip_count
    FROM pickup_zones
    GROUP BY pickup_zone
    ORDER BY trip_count ASC
    LIMIT 10
""")
result.show()

+--------------------+----------+
|         pickup_zone|trip_count|
+--------------------+----------+
|       Arden Heights|         1|
|Eltingville/Annad...|         1|
|Governor's Island...|         1|
|       Port Richmond|         3|
| Green-Wood Cemetery|         4|
|         Great Kills|         4|
|       Rikers Island|         4|
|   Rossville/Woodrow|         4|
|         Jamaica Bay|         5|
|         Westerleigh|        12|
+--------------------+----------+

